In [233]:
import gymnasium as gym
import numpy as np
import pandas as pd
import random

In [234]:
epsilon = 1.0
epsilon_decay = 0.9995
min_epsilon = 0.01
num_episodes = 10000 # this number added because there are nearly 9000 iterations that can go
max_steps = 100 # max steps on this can go up to only 200, i added 100 to save the time

In [235]:
# init the environment
env = gym.make("Taxi-v4")

# action types
actions = ["south(0)", "north(1)", "east(2)", "west(3)", "pickup(4)", "dropoff(5)"]

In [236]:
# init q table (500 states x 6 actions)
q_table = np.zeros((env.observation_space.n, env.action_space.n))

# display initial q table with initial 0 values
actions = ["south(0)", "north(1)", "east(2)", "west(3)", "pickup(4)", "dropoff(5)"]
q_df = pd.DataFrame(q_table, columns=actions)
q_df.index.name = "state"
q_df


,south(0),north(1),east(2),west(3),pickup(4),dropoff(5)
state,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
495,0.0,0.0,0.0,0.0,0.0,0.0
496,0.0,0.0,0.0,0.0,0.0,0.0
497,0.0,0.0,0.0,0.0,0.0,0.0


In [237]:
# helping functions
def get_observed_q_value(reward, gamma, result_state_highest_q_value_of_action, terminated):
    if terminated:
        return reward
    return reward + (gamma * result_state_highest_q_value_of_action)

def get_td_error(observed_q_value, existing_q_value):
    return observed_q_value - existing_q_value

def update_rule_of_cell(existing_q_value, alpha, td_error):
    return existing_q_value + (alpha * td_error)

def choose_action_greedy(q_table, state, epsilon):
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    return int(np.argmax(q_table[state]))

In [238]:
# update one cell at a time
def update_one_cell(Q, state, action, reward, next_state, terminated, alpha, gamma):
    result_state_highest_q_value_of_action = np.max(Q[next_state])
    existing_q_value = Q[state, action]

    observed_q_value = get_observed_q_value(reward, gamma, result_state_highest_q_value_of_action, terminated)
    td_error = get_td_error(observed_q_value, existing_q_value)

    Q[state, action] = update_rule_of_cell(existing_q_value, alpha, td_error)

def run_one_episode(Q, epsilon, alpha, gamma, max_steps, reset_seed=None):
    # reset_seed is only given on the first episode; None means "carry on the sequence"
    state, _ = env.reset(seed=reset_seed)
    total_reward = 0

    for step in range(max_steps):
        action = choose_action_greedy(Q, state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)

        update_one_cell(Q, state, action, reward, next_state, terminated, alpha, gamma)

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return total_reward

In [239]:
# iterate over and learn
def train_q_learning(num_episodes=10000, alpha=0.9, gamma=0.95,
                     epsilon=1.0, epsilon_decay=0.9995, min_epsilon=0.01, max_steps=100, seed=0):
    
    random.seed(seed)              
    env.action_space.seed(seed)
    Q = q_table
    rewards_per_episode = []

    for episode in range(num_episodes):
        # seed the reset only on the first episode; later resets continue the same sequence
        reset_seed = seed if episode == 0 else None
        total_reward = run_one_episode(Q, epsilon, alpha, gamma, max_steps, reset_seed)
        rewards_per_episode.append(total_reward)
        epsilon = max(min_epsilon, epsilon * epsilon_decay)

    return Q, rewards_per_episode

In [240]:
Q, rewards = train_q_learning()

print("Non-zero cells:", np.count_nonzero(Q), "/", Q.size)
print("Average reward, first 100 episodes:", np.mean(rewards[:100]))
print("Average reward, last 100 episodes :", np.mean(rewards[-100:]))

Non-zero cells: 2400 / 3000
Average reward, first 100 episodes: -384.26
Average reward, last 100 episodes : 7.76


In [241]:
Q_df = pd.DataFrame(Q, columns=actions)
Q_df.index.name = "state"
Q_df
# print(Q_df.loc[[314, 328]].round(2))

,south(0),north(1),east(2),west(3),pickup(4),dropoff(5)
state,,,,,,
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2.752004,3.949478,2.752004,3.949478,5.209976,-5.050522
2,7.933492,9.403676,7.933492,9.403676,10.951237,0.403676
3,3.949478,5.209976,3.949478,5.209976,6.536817,-3.790024
4,-3.278345,-4.338712,-3.275187,-4.133447,-13.111633,-13.111732
...,...,...,...,...,...,...
495,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
496,5.209976,6.536817,5.209976,6.536817,-3.790024,-3.790024
497,10.951237,12.580250,10.951237,9.403676,1.951237,1.951237


In [242]:
# extract only the 404 reachable states
def get_reachable_states(env):
    u = env.unwrapped
    reachable_states = []

    for s in range(env.observation_space.n):
        taxi_row, taxi_col, pass_idx, dest_idx = u.decode(s)

        if pass_idx < 4 and pass_idx != dest_idx:        # passenger waiting at a stop
            reachable_states.append(s)
        elif pass_idx == 4:                               # passenger in the taxi
            reachable_states.append(s)
        elif (taxi_row, taxi_col) == u.locs[dest_idx]:    # just dropped off at the destination
            reachable_states.append(s)
        # anything else: passenger at the destination but taxi elsewhere -> impossible

    return reachable_states

In [243]:
# prepare the reachable q_table
def build_reachable_q_table(Q, env):
    u = env.unwrapped
    rows = []

    for s in get_reachable_states(env):
        taxi_row, taxi_col, pass_idx, dest_idx = u.decode(s)
        rows.append([s, taxi_row, taxi_col, pass_idx, dest_idx] + list(Q[s]))

    columns = ["state", "taxi_row", "taxi_col", "pass_idx", "dest_idx"] + actions
    return pd.DataFrame(rows, columns=columns).set_index("state")

In [244]:
reachable_Q_df = build_reachable_q_table(Q, env)

print("Full Q-table     :", Q_df.shape)
print("Reachable Q-table:", reachable_Q_df.shape)
reachable_Q_df

Full Q-table     : (500, 6)
Reachable Q-table: (404, 10)


,taxi_row,taxi_col,pass_idx,dest_idx,south(0),north(1),east(2),west(3),pickup(4),dropoff(5)
state,,,,,,,,,,
0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,0,0,1,2.752004,3.949478,2.752004,3.949478,5.209976,-5.050522
2,0,0,0,2,7.933492,9.403676,7.933492,9.403676,10.951237,0.403676
3,0,0,0,3,3.949478,5.209976,3.949478,5.209976,6.536817,-3.790024
4,0,0,1,0,-3.278345,-4.338712,-3.275187,-4.133447,-13.111633,-13.111732
...,...,...,...,...,...,...,...,...,...,...
494,4,4,3,2,3.833263,2.751910,3.949467,5.209976,-5.060963,-5.060981
496,4,4,4,0,5.209976,6.536817,5.209976,6.536817,-3.790024,-3.790024
497,4,4,4,1,10.951237,12.580250,10.951237,9.403676,1.951237,1.951237


In [245]:
# the states that were left out (impossible states)
reachable_states = get_reachable_states(env)
skipped_states = [s for s in range(env.observation_space.n) if s not in reachable_states]

print("Skipped states:", len(skipped_states))
print(skipped_states)

Skipped states: 96
[5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 90, 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180, 185, 190, 195, 200, 205, 210, 215, 220, 225, 230, 235, 240, 245, 250, 255, 260, 265, 270, 275, 280, 285, 290, 295, 300, 305, 310, 315, 320, 325, 330, 335, 340, 345, 350, 355, 360, 365, 370, 375, 380, 385, 390, 395, 400, 405, 415, 420, 425, 430, 435, 440, 445, 450, 455, 460, 465, 470, 480, 485, 490, 495]


In [246]:
# adding the best action coloumn for the table followed by the state (state, taxi_row, taxi_col, pass_idx, dest_idx, best_action)
reachable_Q_df["best_action"] = reachable_Q_df[actions].values.argmax(axis=1)

# keep only the inputs and the label (the Q-values are not inputs)
dataset_df = reachable_Q_df[["taxi_row", "taxi_col", "pass_idx", "dest_idx", "best_action"]]

print(dataset_df)
print("How many times each action is the best action:")
print(dataset_df["best_action"].value_counts().sort_index())

# the 4 finished states have all-zero Q-rows, so their label (0) is arbitrary
print("Finished states:")
print(dataset_df.loc[[0, 85, 410, 475]])

dataset_df.to_csv("taxi_dataset.csv")

       taxi_row  taxi_col  pass_idx  dest_idx  best_action
state                                                     
0             0         0         0         0            0
1             0         0         0         1            4
2             0         0         0         2            4
3             0         0         0         3            4
4             0         0         1         0            2
...         ...       ...       ...       ...          ...
494           4         4         3         2            3
496           4         4         4         0            1
497           4         4         4         1            1
498           4         4         4         2            1
499           4         4         4         3            3

[404 rows x 5 columns]
How many times each action is the best action:
best_action
0    117
1    152
2     47
3     72
4     12
5      4
Name: count, dtype: int64
Finished states:
       taxi_row  taxi_col  pass_idx  dest_idx  best_a